Environment and imports
Load and inspect data (data/training_data.csv)
Build next-day return / direction target
Basic preprocessing and missing value handling
Normalize / scale features (and reproduce project normalizers if needed)
Correlation metrics (point-biserial, Spearman) and ranking
Visualizations (feature distributions by direction, correlation heatmap)
Per-feature statistical tests (t-test or Mann–Whitney)
Feature importance via models (logistic regression, RF, permutation)
Cross-validated predictive check (AUC, precision/recall)
Save/export ranked features and plots

## Step 1 — Environment & Imports ✅

This section sets up the analysis environment and imports the libraries we'll use for exploration, visualization and modeling. If a package is missing, install it in your environment (e.g., `pip install pandas seaborn scikit-learn matplotlib scipy`).

In [ ]:
# Step 1: Imports and plotting defaults
%pip install pandas numpy matplotlib seaborn scipy scikit-learn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve, confusion_matrix, classification_report
import warnings

warnings.filterwarnings("ignore")

sns.set(style='whitegrid', context='notebook', rc={'figure.figsize':(10,6)})
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"pandas={pd.__version__}, numpy={np.__version__}, seaborn={sns.__version__}")

## Step 2 — Load & Inspect data/training_data.csv ✅

This section loads the dataset and prints a quick inspection (shape, dtypes, missing values, sample rows). It will also try to parse date columns and show basic numeric summaries.

In [ ]:
# Step 2: Load and inspect training data
from pathlib import Path
import os

# Change to workspace root if not already there
workspace_root = Path("/Users/alex/rust_llm_stock")
if Path.cwd() != workspace_root:
    os.chdir(workspace_root)

# Update this path to match your actual exported data file
# Try the newly exported file first, then fall back
DATA_PATH = Path("./data/zb_training_jan_jun.csv")
if not DATA_PATH.exists():
    # Fall back to other exports
    import glob
    csv_files = sorted(glob.glob("./data/zb_training*.csv"), reverse=True)
    if csv_files:
        DATA_PATH = Path(csv_files[0])
        print(f"Found: {DATA_PATH}")
    else:
        print("⚠️ No training data CSV found in ./data/")

print(f"Loading: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)

print(f"\n📊 Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]}... ({len(df.columns)} total)")
print(f"\nDtypes:\n{df.dtypes.value_counts()}")

# Identify column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = {'ts_code', 'trade_date', 'next_day_direction', 'next_3day_direction', 'next_day_return', 'next_3day_return'}
exclude_prefixes = tuple(['industry_emb', 'act_ent_type_emb'])
model_feature_cols = [c for c in num_cols if c not in exclude_cols and not any(c.startswith(p) for p in exclude_prefixes)]
label_cols = [c for c in df.columns if 'direction' in c.lower() or 'return' in c.lower()]
temporal_cols = [c for c in df.columns if any(t in c.lower() for t in ['trade_date', 'ts_code'])]

print(f"\n🔍 Column types:")
print(f"  Numeric features: {len(model_feature_cols)}")
print(f"  ID/Temporal columns: {len(temporal_cols)}")
print(f"  Label columns: {len(label_cols)}")
print(f"  Embeddings: {len([c for c in num_cols if any(c.startswith(p) for p in exclude_prefixes)])}")

# Basic missing data check
rows_with_missing = df[model_feature_cols].isnull().any(axis=1).sum()
EXPECTED_FEATURE_SIZE = len(model_feature_cols)
print(f"\n✅ Rows with any missing values: {rows_with_missing} ({rows_with_missing/len(df)*100:.3f}%)")
print(f"Total features for model: {EXPECTED_FEATURE_SIZE}")

# Show sample
print(f"\nSample rows:")
display(df.iloc[:3, :5])

### Dataset creator fix applied ✅

- **What I changed**: extended industry-performance prefetch to include a short backward buffer (to support "prior-day" lookups) and added the industry feature columns to the dynamic batch INSERT so they are persisted.
- **Verification performed**: ran `dataset_creator` for 2024-04-01 → 2025-12-31 (inserted 2,242,357 rows) and exported `./data/training_data.csv` (204,684 rows for prefix `60`).

**What to do now**: run the verification cell below to check missing counts for the previously-missing fields; if all good, proceed to model training.


# Data quality diagnostics for ../data/training_data_2025.csv

This cell loads the training data for 2025 and runs a set of checks useful before model training:

- Basic shape, dtypes, and sample rows
- Missingness counts and percentages (top columns)
- Numeric summary (percentiles, zeros %, missing %)
- Constant columns, duplicates, and label distribution
- Outlier checks and a sample correlation heatmap
- Save diagnostics to `artifacts/diagnostics_training_data_2025.json` and `artifacts/missingness_training_data_2025.csv`

Run this cell to produce a quick data-quality report and visuals.

In [ ]:
# Advanced Training Data Diagnostics
# This cell provides comprehensive analysis of data quality and ML-readiness

print("="*70)
print("🔍 TRAINING DATA DIAGNOSTICS & DISTRIBUTION ANALYSIS")
print("="*70)

# === 1. CLASS BALANCE ANALYSIS ===
print("\n" + "="*70)
print("1. TARGET CLASS DISTRIBUTION")
print("="*70)

target_cols = [c for c in df.columns if 'direction' in c.lower() or c in ['target', 'label', 'y']]
for target in target_cols:
    if target in df.columns:
        print(f"\n📊 {target}:")
        dist = df[target].value_counts(dropna=False).sort_index()
        dist_pct = df[target].value_counts(dropna=False, normalize=True).sort_index() * 100
        
        result = pd.DataFrame({
            'count': dist,
            'percentage': dist_pct
        })
        display(result)
        
        # Class imbalance check
        if dist.min() > 0:
            balance_ratio = dist.min() / dist.max()
            if balance_ratio < 0.3:
                print(f"   ⚠️  IMBALANCED: ratio {balance_ratio:.3f} (min/max). Consider SMOTE or class_weight.")
            elif balance_ratio < 0.5:
                print(f"   ⚙️  Moderate imbalance: ratio {balance_ratio:.3f}. Use stratified splits.")
            else:
                print(f"   ✅ Well balanced: ratio {balance_ratio:.3f}")

# === 2. FEATURE CORRELATION & MULTICOLLINEARITY ===
print("\n" + "="*70)
print("2. HIGH CORRELATION DETECTION (Multicollinearity)")
print("="*70)

# Get model features only
model_feats = [c for c in num_cols if c not in exclude_cols and not any(c.startswith(p) for p in exclude_prefixes)]
if len(model_feats) > 1:
    # Sample for speed
    corr_sample = df[model_feats].sample(n=min(10000, len(df)), random_state=42)
    corr_matrix = corr_sample.corr().abs()
    
    # Find highly correlated pairs (exclude diagonal)
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = [(col, row, upper_tri.loc[row, col]) 
                       for col in upper_tri.columns 
                       for row in upper_tri.index 
                       if upper_tri.loc[row, col] > 0.9]
    
    if high_corr_pairs:
        print(f"⚠️  Found {len(high_corr_pairs)} feature pairs with correlation > 0.9:")
        high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature_1', 'Feature_2', 'Correlation'])
        display(high_corr_df.sort_values('Correlation', ascending=False).head(20))
        print("   💡 ACTION: Consider removing one from each pair to reduce multicollinearity.")
    else:
        print("✅ No severe multicollinearity detected (threshold: 0.9)")

# === 3. DATA LEAKAGE DETECTION ===
print("\n" + "="*70)
print("3. POTENTIAL DATA LEAKAGE CHECKS")
print("="*70)

leakage_issues = []

# Check for future-looking features
future_keywords = ['next', 'future', 'forward', 'ahead', 'tomorrow']
suspicious_feats = [c for c in model_feats if any(kw in c.lower() for kw in future_keywords)]
if suspicious_feats:
    leakage_issues.append(f"⚠️  Suspicious future-looking features: {suspicious_feats[:10]}")

# Check for perfect correlations with targets
for target in target_cols:
    if target in df.columns and not df[target].isnull().all():
        for feat in model_feats[:50]:  # Check first 50 for speed
            try:
                corr = df[[feat, target]].dropna().corr().iloc[0, 1]
                if abs(corr) > 0.95:
                    leakage_issues.append(f"⚠️  '{feat}' has {corr:.3f} correlation with target '{target}' - potential leakage!")
            except:
                pass

# Temporal leakage check
temporal_feats = [c for c in model_feats if any(t in c.lower() for t in ['month', 'weekday', 'quarter', 'week_no', 'year', 'day_of'])]
if temporal_feats:
    leakage_issues.append(f"⚠️  Temporal features present: {temporal_feats}. Risk of leakage if test data has different time distribution.")

if leakage_issues:
    for issue in leakage_issues:
        print(issue)
else:
    print("✅ No obvious data leakage detected")

# === 4. FEATURE VARIANCE & INFORMATION CONTENT ===
print("\n" + "="*70)
print("4. LOW-VARIANCE & ZERO-INFORMATION FEATURES")
print("="*70)

variance_check = df[model_feats].var().sort_values()
low_var_feats = variance_check[variance_check < 1e-6].index.tolist()

if low_var_feats:
    print(f"⚠️  {len(low_var_feats)} features with very low variance (< 1e-6):")
    print(f"   {low_var_feats[:20]}")
    print("   💡 ACTION: These provide little information. Consider removing.")
else:
    print("✅ All features have sufficient variance")

# Check near-zero variance (>95% same value)
nzv_feats = []
for feat in model_feats:
    if df[feat].notnull().sum() > 0:
        mode_freq = df[feat].value_counts().iloc[0] if len(df[feat].value_counts()) > 0 else 0
        if mode_freq / len(df) > 0.95:
            nzv_feats.append((feat, mode_freq / len(df)))

if nzv_feats:
    print(f"\n⚠️  {len(nzv_feats)} features with >95% same value:")
    for feat, ratio in nzv_feats[:10]:
        print(f"   {feat}: {ratio*100:.1f}%")
else:
    print("✅ No near-zero variance features")

# === 5. OUTLIER IMPACT ANALYSIS ===
print("\n" + "="*70)
print("5. EXTREME OUTLIERS (>3 IQR)")
print("="*70)

extreme_outliers = {}
for feat in model_feats[:30]:  # Sample for speed
    q1 = df[feat].quantile(0.25)
    q3 = df[feat].quantile(0.75)
    iqr = q3 - q1
    if iqr > 0:
        outlier_mask = (df[feat] < (q1 - 3*iqr)) | (df[feat] > (q3 + 3*iqr))
        outlier_pct = outlier_mask.sum() / len(df) * 100
        if outlier_pct > 1:
            extreme_outliers[feat] = outlier_pct

if extreme_outliers:
    print(f"⚠️  {len(extreme_outliers)} features with >1% extreme outliers:")
    outlier_df = pd.DataFrame.from_dict(extreme_outliers, orient='index', columns=['outlier_pct']).sort_values('outlier_pct', ascending=False)
    display(outlier_df.head(15))
    print("   💡 ACTION: Consider Winsorization or robust scaling (RobustScaler).")
else:
    print("✅ Low outlier prevalence across features")

# === 6. MISSING DATA PATTERN ANALYSIS ===
print("\n" + "="*70)
print("6. MISSING DATA PATTERNS")
print("="*70)

cols_with_missing = [c for c in model_feats if df[c].isnull().sum() > 0]
if cols_with_missing:
    print(f"⚠️  {len(cols_with_missing)} features have missing values")
    
    # Check if missingness is random or patterned
    if len(cols_with_missing) > 1:
        missing_corr = df[cols_with_missing].isnull().corr()
        high_missing_corr = []
        for i in range(len(missing_corr.columns)):
            for j in range(i+1, len(missing_corr.columns)):
                if missing_corr.iloc[i, j] > 0.7:
                    high_missing_corr.append((missing_corr.columns[i], missing_corr.columns[j], missing_corr.iloc[i, j]))
        
        if high_missing_corr:
            print(f"\n⚠️  {len(high_missing_corr)} pairs of features with correlated missingness (>0.7):")
            for f1, f2, corr in high_missing_corr[:10]:
                print(f"   {f1} <-> {f2}: {corr:.3f}")
            print("   💡 This suggests systematic missing data (e.g., missing for certain stock types)")
        else:
            print("✅ Missingness appears random (low correlation between missing patterns)")
else:
    print("✅ No missing values in model features")

# === 7. SUMMARY & RECOMMENDATIONS ===
print("\n" + "="*70)
print("📋 OPTIMIZATION SUMMARY")
print("="*70)

issues_found = []
actions = []

if len(model_feats) != EXPECTED_FEATURE_SIZE:
    issues_found.append(f"Feature count: {len(model_feats)} vs expected {EXPECTED_FEATURE_SIZE}")
    actions.append("Fix export script to match expected feature count")

if rows_with_missing > 0 and rows_with_missing < len(df) * 0.01:
    issues_found.append(f"Missing data: {rows_with_missing} rows ({rows_with_missing/len(df)*100:.3f}%)")
    actions.append("Drop missing rows (< 1% impact)")

if high_corr_pairs:
    issues_found.append(f"Multicollinearity: {len(high_corr_pairs)} highly correlated pairs")
    actions.append("Remove redundant features")

if low_var_feats or nzv_feats:
    issues_found.append(f"Low information features: {len(low_var_feats) + len(nzv_feats)}")
    actions.append("Remove near-zero variance features")

if extreme_outliers:
    issues_found.append(f"Extreme outliers in {len(extreme_outliers)} features")
    actions.append("Apply Winsorization or robust scaling")

if leakage_issues:
    issues_found.append("Potential data leakage detected")
    actions.append("Review suspicious features and temporal metadata")

print(f"\n🔍 Issues detected: {len(issues_found)}")
if issues_found:
    for i, issue in enumerate(issues_found, 1):
        print(f"  {i}. {issue}")
    print(f"\n💡 Recommended actions:")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")
else:
    print("✅ Dataset appears well-prepared for training!")

print("\n" + "="*70)


## Step 3 — Outlier Detection & Visualization 📊

This section analyzes and visualizes outliers in the training data using multiple methods:
- **IQR Method**: Identifies values beyond 1.5 × IQR or 3 × IQR
- **Z-Score Method**: Flags values more than 3 standard deviations from mean
- **Boxplots**: Visual inspection of distribution and extreme values
- **Histograms**: Distribution shape and outlier impact


In [ ]:
# === OUTLIER DETECTION & VISUALIZATION ===

print("="*70)
print("🔍 COMPREHENSIVE OUTLIER ANALYSIS")
print("="*70)

# Prepare feature list
model_feats = [c for c in num_cols if c not in exclude_cols and not any(c.startswith(p) for p in exclude_prefixes)]

# === Method 1: IQR-based Outlier Detection ===
print("\n1️⃣  IQR METHOD (Mild & Extreme Outliers)")
print("-" * 70)

iqr_outliers = {}
for feat in model_feats:
    q1 = df[feat].quantile(0.25)
    q3 = df[feat].quantile(0.75)
    iqr = q3 - q1
    
    if iqr > 0:
        # Mild outliers: 1.5 × IQR
        mild_lower = q1 - 1.5 * iqr
        mild_upper = q3 + 1.5 * iqr
        mild_mask = (df[feat] < mild_lower) | (df[feat] > mild_upper)
        mild_count = mild_mask.sum()
        mild_pct = mild_count / len(df) * 100
        
        # Extreme outliers: 3 × IQR
        extreme_lower = q1 - 3 * iqr
        extreme_upper = q3 + 3 * iqr
        extreme_mask = (df[feat] < extreme_lower) | (df[feat] > extreme_upper)
        extreme_count = extreme_mask.sum()
        extreme_pct = extreme_count / len(df) * 100
        
        if mild_count > 0 or extreme_count > 0:
            iqr_outliers[feat] = {
                'mild_count': mild_count,
                'mild_pct': mild_pct,
                'extreme_count': extreme_count,
                'extreme_pct': extreme_pct,
                'q1': q1,
                'q3': q3,
                'iqr': iqr
            }

# Sort by extreme outlier percentage and display top features
top_iqr = sorted(iqr_outliers.items(), key=lambda x: x[1]['extreme_pct'], reverse=True)[:15]

outlier_summary = []
for feat, stats in top_iqr:
    outlier_summary.append({
        'Feature': feat,
        'Mild Outliers (1.5×IQR)': f"{stats['mild_count']} ({stats['mild_pct']:.2f}%)",
        'Extreme Outliers (3×IQR)': f"{stats['extreme_count']} ({stats['extreme_pct']:.2f}%)",
        'Q1': f"{stats['q1']:.4f}",
        'Q3': f"{stats['q3']:.4f}",
        'IQR': f"{stats['iqr']:.4f}"
    })

outlier_df_iqr = pd.DataFrame(outlier_summary)
print("\n✅ Top 15 features with most extreme outliers (3×IQR):\n")
display(outlier_df_iqr)

# === Method 2: Z-Score Method ===
print("\n2️⃣  Z-SCORE METHOD (>3 std dev)")
print("-" * 70)

zscore_outliers = {}
for feat in model_feats:
    if df[feat].notna().sum() > 0:
        mean = df[feat].mean()
        std = df[feat].std()
        
        if std > 0:
            z_scores = np.abs((df[feat] - mean) / std)
            outlier_mask = z_scores > 3
            outlier_count = outlier_mask.sum()
            outlier_pct = outlier_count / len(df) * 100
            
            if outlier_count > 0:
                zscore_outliers[feat] = {
                    'count': outlier_count,
                    'pct': outlier_pct,
                    'mean': mean,
                    'std': std,
                    'max_z': z_scores.max()
                }

top_zscore = sorted(zscore_outliers.items(), key=lambda x: x[1]['pct'], reverse=True)[:15]

zscore_summary = []
for feat, stats in top_zscore:
    zscore_summary.append({
        'Feature': feat,
        'Outliers (>3σ)': f"{stats['count']} ({stats['pct']:.2f}%)",
        'Mean': f"{stats['mean']:.4f}",
        'Std': f"{stats['std']:.4f}",
        'Max Z-Score': f"{stats['max_z']:.2f}"
    })

zscore_df = pd.DataFrame(zscore_summary)
print("\n✅ Top 15 features with most extreme z-score outliers:\n")
display(zscore_df)

# === Visualization: Boxplots for Top Outlier Features ===
print("\n3️⃣  BOXPLOT VISUALIZATION")
print("-" * 70)

# Get top 12 features with extreme outliers
top_features = [feat for feat, _ in top_iqr[:12]]

if top_features:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, feat in enumerate(top_features):
        ax = axes[idx]
        
        # Create boxplot
        box_data = df[feat].dropna()
        bp = ax.boxplot(box_data, vert=True, patch_artist=True)
        
        # Customize colors
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
        bp['medians'][0].set_color('red')
        bp['medians'][0].set_linewidth(2)
        
        # Add outlier count
        q1 = box_data.quantile(0.25)
        q3 = box_data.quantile(0.75)
        iqr = q3 - q1
        outlier_count = ((box_data < (q1 - 1.5*iqr)) | (box_data > (q3 + 1.5*iqr))).sum()
        
        ax.set_title(f'{feat}\n({outlier_count} outliers)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Value', fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(top_features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('artifacts/outliers_boxplots.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Boxplots saved to artifacts/outliers_boxplots.png")

# === Visualization: Distributions with Outliers Highlighted ===
print("\n4️⃣  HISTOGRAM VISUALIZATION WITH OUTLIER HIGHLIGHTING")
print("-" * 70)

top_8_features = [feat for feat, _ in top_iqr[:8]]

if top_8_features:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, feat in enumerate(top_8_features):
        ax = axes[idx]
        data = df[feat].dropna()
        
        # Calculate outlier bounds
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Split data into normal and outlier
        normal_data = data[(data >= lower_bound) & (data <= upper_bound)]
        outlier_data = data[(data < lower_bound) | (data > upper_bound)]
        
        # Plot
        ax.hist(normal_data, bins=30, alpha=0.7, color='blue', label='Normal')
        if len(outlier_data) > 0:
            ax.hist(outlier_data, bins=10, alpha=0.7, color='red', label='Outliers')
        
        ax.set_title(f'{feat}\nOutliers: {len(outlier_data)} ({len(outlier_data)/len(data)*100:.1f}%)', 
                    fontsize=10, fontweight='bold')
        ax.set_xlabel('Value', fontsize=9)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('artifacts/outliers_distributions.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Distributions saved to artifacts/outliers_distributions.png")

# === Summary Statistics ===
print("\n5️⃣  OUTLIER SUMMARY STATISTICS")
print("-" * 70)

total_mild = sum(s['mild_count'] for s in iqr_outliers.values())
total_extreme = sum(s['extreme_count'] for s in iqr_outliers.values())
features_with_outliers = len(iqr_outliers)

print(f"\n📊 Overall Statistics:")
print(f"  • Features with outliers: {features_with_outliers} / {len(model_feats)}")
print(f"  • Total mild outliers (1.5×IQR): {total_mild:,} ({total_mild/(len(df)*len(model_feats))*100:.3f}%)")
print(f"  • Total extreme outliers (3×IQR): {total_extreme:,} ({total_extreme/(len(df)*len(model_feats))*100:.3f}%)")

print(f"\n🎯 Feature with most outliers:")
if top_iqr:
    top_feat, top_stats = top_iqr[0]
    print(f"  • {top_feat}: {top_stats['extreme_count']} extreme outliers ({top_stats['extreme_pct']:.2f}%)")

print(f"\n💡 Recommendations:")
if total_extreme / (len(df) * len(model_feats)) > 0.05:
    print(f"  ✅ Winsorization or robust scaling recommended (>0.5% extreme outliers)")
else:
    print(f"  ✅ Outlier levels are moderate; StandardScaler should work well")

print("\n" + "="*70)


## Step 4 — Deep Data Quality Assessment 🔬

Advanced diagnostics to evaluate dataset integrity, consistency, and ML-readiness:
- **Duplicates & Uniqueness**: Identify duplicate rows and feature consistency
- **Distribution Analysis**: Skewness, kurtosis, normality tests (Shapiro-Wilk)
- **Data Integrity**: NULL patterns, range violations, type consistency
- **Feature Stability**: Coefficient of variation, inter-quartile ranges
- **Temporal Patterns**: Time-series consistency and date coverage
- **ML Readiness Score**: Composite quality score for model training


In [ ]:
# === DEEP DATA QUALITY ASSESSMENT ===

print("="*70)
print("🔬 COMPREHENSIVE DATA QUALITY EVALUATION")
print("="*70)

from scipy.stats import shapiro, skew, kurtosis

# === 1. DUPLICATES & UNIQUENESS ===
print("\n" + "="*70)
print("1. DUPLICATES & UNIQUENESS ANALYSIS")
print("="*70)

# Full row duplicates
full_duplicates = df.duplicated().sum()
full_dup_pct = (full_duplicates / len(df)) * 100

print(f"\n📋 Full Row Duplicates:")
print(f"  • Count: {full_duplicates} ({full_dup_pct:.3f}%)")

# Check if duplicates by key columns (ts_code + trade_date)
if 'ts_code' in df.columns and 'trade_date' in df.columns:
    key_duplicates = df.duplicated(subset=['ts_code', 'trade_date']).sum()
    key_dup_pct = (key_duplicates / len(df)) * 100
    print(f"  • Key duplicates (ts_code + trade_date): {key_duplicates} ({key_dup_pct:.3f}%)")
    
    if key_duplicates > 0:
        print(f"    ⚠️  WARNING: Duplicate (stock, date) pairs detected - data integrity issue!")
    else:
        print(f"    ✅ No duplicate (stock, date) pairs - key uniqueness maintained")

# Feature value uniqueness
print(f"\n📊 Feature Value Diversity:")
unique_counts = df[model_feats].nunique()
low_diversity_feats = unique_counts[unique_counts < 5].index.tolist()

if low_diversity_feats:
    print(f"  ⚠️  {len(low_diversity_feats)} features with <5 unique values:")
    for feat in low_diversity_feats[:10]:
        print(f"     {feat}: {unique_counts[feat]} unique values")
else:
    print(f"  ✅ All features have sufficient diversity (≥5 unique values)")

# === 2. DISTRIBUTION ANALYSIS ===
print("\n" + "="*70)
print("2. DISTRIBUTION ANALYSIS (Normality & Skewness)")
print("="*70)

distribution_stats = []
non_normal_feats = 0
skewed_feats = 0

for feat in model_feats[:50]:  # Sample for speed
    data = df[feat].dropna()
    
    if len(data) > 3:  # Need at least 3 values for statistics
        # Skewness & Kurtosis
        feat_skew = skew(data)
        feat_kurt = kurtosis(data)
        feat_mean = data.mean()
        feat_std = data.std()
        feat_cv = (feat_std / abs(feat_mean)) if feat_mean != 0 else np.inf
        
        # Shapiro-Wilk normality test (sample if > 5000)
        if len(data) > 5000:
            sample_data = np.random.choice(data, 5000, replace=False)
        else:
            sample_data = data
        
        try:
            stat, p_value = shapiro(sample_data)
            is_normal = p_value > 0.05
        except:
            is_normal = None
        
        if not is_normal:
            non_normal_feats += 1
        
        if abs(feat_skew) > 1:
            skewed_feats += 1
        
        distribution_stats.append({
            'Feature': feat,
            'Skewness': f"{feat_skew:.3f}",
            'Kurtosis': f"{feat_kurt:.3f}",
            'CV (Var/Mean)': f"{feat_cv:.3f}",
            'Normal': '✅' if is_normal else '❌'
        })

if distribution_stats:
    dist_df = pd.DataFrame(distribution_stats)
    print(f"\n✅ Distribution Stats (first 50 features sampled):")
    display(dist_df.head(20))
    print(f"\n📊 Summary:")
    print(f"  • Non-normal features: {non_normal_feats}/{len(distribution_stats)} ({non_normal_feats/len(distribution_stats)*100:.1f}%)")
    print(f"  • Highly skewed features (|skew|>1): {skewed_feats}/{len(distribution_stats)}")
    print(f"  💡 Most features are non-normal - this is common in financial data. Use StandardScaler or RobustScaler.")

# === 3. DATA INTEGRITY CHECKS ===
print("\n" + "="*70)
print("3. DATA INTEGRITY & CONSISTENCY")
print("="*70)

integrity_issues = []

# Check for infinite values
inf_cols = []
for feat in model_feats:
    if np.isinf(df[feat]).sum() > 0:
        inf_count = np.isinf(df[feat]).sum()
        inf_cols.append((feat, inf_count))

if inf_cols:
    print(f"\n⚠️  CRITICAL: {len(inf_cols)} features contain INFINITE values:")
    for feat, count in inf_cols[:10]:
        print(f"   {feat}: {count} infinite values")
        integrity_issues.append(f"Infinite values in {feat}")
else:
    print(f"\n✅ No infinite values detected")

# Check for NaN patterns
nan_by_feature = df[model_feats].isnull().sum().sort_values(ascending=False)
nan_by_feature = nan_by_feature[nan_by_feature > 0]

if len(nan_by_feature) > 0:
    print(f"\n⚠️  {len(nan_by_feature)} features have missing values:")
    nan_df = pd.DataFrame({
        'Feature': nan_by_feature.index,
        'Missing_Count': nan_by_feature.values,
        'Missing_Pct': (nan_by_feature.values / len(df) * 100).round(3)
    })
    display(nan_df.head(15))
else:
    print(f"\n✅ No missing values in model features")

# Check for negative values in expected positive features
positive_keywords = ['price', 'volume', 'amount']
negative_in_positive = []

for feat in model_feats:
    if any(kw in feat.lower() for kw in positive_keywords):
        neg_count = (df[feat] < 0).sum()
        if neg_count > 0:
            neg_pct = (neg_count / len(df)) * 100
            negative_in_positive.append((feat, neg_count, neg_pct))

if negative_in_positive:
    print(f"\n⚠️  {len(negative_in_positive)} 'positive' features have negative values:")
    for feat, count, pct in negative_in_positive[:10]:
        print(f"   {feat}: {count} negative values ({pct:.2f}%)")
else:
    print(f"\n✅ No unexpected negative values in positive-domain features")

# === 4. FEATURE STABILITY & CONSISTENCY ===
print("\n" + "="*70)
print("4. FEATURE STABILITY & CONSISTENCY")
print("="*70)

stability_metrics = []

for feat in model_feats[:40]:  # Sample
    data = df[feat].dropna()
    
    if len(data) > 10:
        # Coefficient of Variation (std/mean) - measure of relative variability
        mean = data.mean()
        std = data.std()
        cv = (std / abs(mean)) if mean != 0 else np.inf
        
        # Inter-quartile range (IQR/median) - robustness measure
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        median = data.median()
        iqr = q3 - q1
        iqr_ratio = (iqr / abs(median)) if median != 0 else np.inf
        
        # Range
        data_range = data.max() - data.min()
        range_ratio = (data_range / abs(mean)) if mean != 0 else np.inf
        
        stability_metrics.append({
            'Feature': feat,
            'CV': f"{cv:.3f}",
            'IQR/Median': f"{iqr_ratio:.3f}",
            'Range/Mean': f"{range_ratio:.3f}",
            'Stability': 'HIGH' if cv < 0.5 else ('MEDIUM' if cv < 2.0 else 'LOW')
        })

if stability_metrics:
    stab_df = pd.DataFrame(stability_metrics)
    print(f"\n✅ Feature Stability Metrics (first 40 features):")
    display(stab_df.head(15))
    
    high_var_feats = stab_df[stab_df['Stability'] == 'LOW']
    print(f"\n💡 Features with LOW stability (high variance): {len(high_var_feats)}")

# === 5. TEMPORAL PATTERNS (if date columns exist) ===
print("\n" + "="*70)
print("5. TEMPORAL PATTERNS & DATE COVERAGE")
print("="*70)

if 'trade_date' in df.columns:
    df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d', errors='coerce')
    
    # Date range
    min_date = df['trade_date'].min()
    max_date = df['trade_date'].max()
    date_range_days = (max_date - min_date).days
    
    print(f"\n📅 Temporal Coverage:")
    print(f"  • Date range: {min_date.date()} to {max_date.date()}")
    print(f"  • Total days: {date_range_days}")
    print(f"  • Expected trading days (≈250/year): {date_range_days * 250 / 365:.0f}")
    
    # Rows per date
    rows_per_date = df['trade_date'].value_counts()
    print(f"\n  • Rows per date:")
    print(f"    - Min: {rows_per_date.min()}")
    print(f"    - Max: {rows_per_date.max()}")
    print(f"    - Mean: {rows_per_date.mean():.1f}")
    print(f"    - Std: {rows_per_date.std():.1f}")
    
    # Date gaps
    unique_dates = sorted(df['trade_date'].unique())
    if len(unique_dates) > 1:
        date_diffs = [(unique_dates[i+1] - unique_dates[i]).days for i in range(len(unique_dates)-1)]
        max_gap = max(date_diffs)
        if max_gap > 7:
            print(f"\n  ⚠️  Large date gap detected: {max_gap} days")
        else:
            print(f"\n  ✅ Date coverage is consistent (max gap: {max_gap} days)")
    
    # Stocks per date
    if 'ts_code' in df.columns:
        stocks_per_date = df.groupby('trade_date')['ts_code'].nunique()
        print(f"\n  • Stocks per date:")
        print(f"    - Min: {stocks_per_date.min()}")
        print(f"    - Max: {stocks_per_date.max()}")
        print(f"    - Mean: {stocks_per_date.mean():.1f}")

# === 6. ML READINESS SCORE ===
print("\n" + "="*70)
print("6. ML READINESS SCORE")
print("="*70)

readiness_score = 100.0
readiness_issues = []

# Check various quality metrics
if full_dup_pct > 0:
    readiness_score -= min(10, full_dup_pct)
    readiness_issues.append(f"Full duplicates: {full_dup_pct:.2f}%")

if len(nan_by_feature) > len(model_feats) * 0.3:
    readiness_score -= 15
    readiness_issues.append("High missing value prevalence")

if len(extreme_outliers) > len(model_feats) * 0.5:
    readiness_score -= 10
    readiness_issues.append("Many features with extreme outliers")

if len(low_diversity_feats) > 0:
    readiness_score -= 5
    readiness_issues.append("Low diversity in some features")

if len(inf_cols) > 0:
    readiness_score -= 20
    readiness_issues.append("CRITICAL: Infinite values detected")

if len(high_corr_pairs) > len(model_feats) * 0.1:
    readiness_score -= 10
    readiness_issues.append("High multicollinearity detected")

readiness_score = max(0, readiness_score)

print(f"\n📈 ML READINESS SCORE: {readiness_score:.1f}/100")

if readiness_score >= 80:
    print("   ✅ EXCELLENT - Ready for ML training")
elif readiness_score >= 60:
    print("   ⚙️  GOOD - Minor preprocessing needed")
elif readiness_score >= 40:
    print("   ⚠️  FAIR - Significant preprocessing required")
else:
    print("   🔴 POOR - Major data quality issues")

if readiness_issues:
    print(f"\n💡 Issues affecting readiness score:")
    for i, issue in enumerate(readiness_issues, 1):
        print(f"   {i}. {issue}")

print(f"\n📋 Recommendations for ML Training:")
print(f"   1. Handle missing values: {'SimpleImputer or drop rows' if len(nan_by_feature) > 0 else 'N/A'}")
print(f"   2. Scaling strategy: StandardScaler or RobustScaler (for robustness against outliers)")
print(f"   3. Feature engineering: Consider removing low-variance or highly correlated features")
print(f"   4. Class balance: {'Apply SMOTE if needed' if min(df[target_cols].value_counts()) / max(df[target_cols].value_counts()) < 0.5 else 'Balanced classes'}")
print(f"   5. Cross-validation: Use StratifiedKFold to maintain class distribution in folds")

print("\n" + "="*70)
